In [1]:
### importing the required libraries
using CSV
using DataFrames
using Statistics
using Plots
using DataStructures  # For the Counter function

In [2]:
## vary these for different dispersal scenarios
gamma = 0.33    # dispersal rate from release site to household
rho = 0.33     # rate visit the households
tau = 0.33      # rate leave the households
alpha = 0.1     # dispersal rate of females moving from household directly to another household

rel_fem = 8
rel_male = 2
delta_rel = 50

num_simulations = 100  # number of realisations of SSA to run, 10,000 is a good number for the paper
rel_type = "relhhold"

"relhhold"

In [3]:
# Define parameters dictionary
fem_w0 = 0  # how many wildtypes and infected mosquitoes starting off with
male_w0 = 0
fem_m0 = 8
male_m0 = 8
m_free0 = 0 # no. of free wild-type
w_free0 = 0 # no. of free Wolbachia

# model parameters
phi = 0.85      # wolbachia fitness effect on birth rate
K = 30          # carrying capacity of mosquitoes per household
d = 12/100      # death rate of mosquitoes
k = 0.3         # larval density parameter
h = 0.19*100^k  # larval density parameter
b = 0.54        # per capita birth rate of mosquitoes (wildtypes)
H = 100         # number of households
u = 1           # vertical transmission probability
v = 1           # CI effect, proportion of non-viable offspring from infected male and wildtype female birth

rel_t = 150   # initial release time

## remember this is per household for the household releases so should use smaller release numbers
## currently set for community wide release

t_start = 0    
t_end = 1000   # start time and end time (days) of simulation

result_length = length(t_start:t_end) # number of time points to store results
weeks = round(Int,result_length/7)    # number of weeks to store results

parameters = Dict(           # dictionary of parameters
    :fem_m0 => fem_m0,
    :male_m0 => male_m0,
    :fem_w0 => fem_w0,
    :male_w0 => male_w0,
    :m_free0 => m_free0,
    :w_free0 => w_free0,
    :rho => rho,
    :phi => phi,
    :b => b,
    :K => K,
    :d => d,
    :h => h,
    :k => k,
    :u => u,
    :v => v,
    :tau => tau,
    :H => H,
    :t_start => t_start,
    :t_end => t_end,
    :seed => 1234,
    :delta_rel => delta_rel,
    #:rel_size => rel_size,
    :gamma => gamma,
    :alpha => alpha,
    :rel_t => rel_t
)

Dict{Symbol, Real} with 24 entries:
  :b       => 0.54
  :alpha   => 0.1
  :gamma   => 0.33
  :m_free0 => 0
  :rho     => 0.33
  :t_start => 0
  :h       => 0.756404
  :rel_t   => 150
  :male_w0 => 0
  :fem_m0  => 8
  :K       => 30
  :phi     => 0.85
  :fem_w0  => 0
  :d       => 0.12
  :k       => 0.3
  :v       => 1
  :u       => 1
  :male_m0 => 8
  :tau     => 0.33
  ⋮        => ⋮

In [4]:
### Convert dispersal parameters to a string and remove the decimal point
## We will use this to call the correct data files
rho_str = replace(string(rho), "." => "")
tau_str = replace(string(tau), "." => "")
gamma_str = replace(string(gamma), "." => "")
alpha_str = replace(string(alpha), "." => "")

"01"

In [5]:
# Read the CSV file into a DataFrame
JOBS_DF = CSV.read("reg_rel_single_hhold.csv", DataFrame)

# Function to find the JOB_ID for given rel_fem and rel_male
function find_job_id(df::DataFrame, rel_fem_value, rel_male_value, rel_time)
    found = false
    for row in eachrow(df)
        if !ismissing(row[Symbol("job no.")]) &&
           row[Symbol("no. females")] == rel_fem_value && row[Symbol("no. males")] == rel_male_value &&
           row[Symbol("release time step")] == rel_time
            return row[Symbol("job no.")]
            found = true
            break
        else
            continue
        end
    end # Return nothing if no matching row is found
    if !found
        return nothing
    end
end

# Example usage
rel_fem_value = 1  # Replace with the desired rel_fem value
rel_male_value = 1  # Replace with the desired rel_male value
rel_time = 10  # Replace with the desired rel_time value

job_id = find_job_id(JOBS_DF, rel_fem_value, rel_male_value, rel_time)
println(job_id)


67140673


In [7]:
using CSV
using DataFrames
using Plots

# Function to count columns with zero
# function count_columns_with_zero(df::DataFrame)
#     count_zeros = 0
#     for col in eachcol(df)
#         if col[end] == 0.0
#             count_zeros += 1
#         end
#     end
#     return count_zeros
# end

function count_columns_with_zero(df::DataFrame)
    count_zeros = 0
    for col in eachcol(df)
        if 0.0 in col
            count_zeros += 1
        end
    end
    return count_zeros
end

# Define the sex ratios and release time steps
sex_ratios = [(1,1), (3,1), (3,3), (3,5), (3,10), (5,1), (5,3), (5,5), (5,10), (10,3), (10,5), (10,10), (10,1)]
rel_time_steps = [100, 75, 50, 25, 10]

# Initialize the dictionary to store ratios
ratios_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

# Loop through each combination of sex ratios and release time steps
for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        if JOB_ID != nothing
            df_m = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "m_mosqs_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            # Calculate the ratio
            ratio = count_columns_with_zero(df_m) / size(df_m, 2)
            # Store the ratio in the dictionary
            ratios_dict[(r, t)] = ratio
            println(ratio)
        end
    end
end

# Print the dictionary
println(ratios_dict)
# Convert the dictionary to a DataFrame
ratios_df = DataFrame(SexRatio=String[], ReleaseTimeStep=Int[], Ratio=Float64[])
for ((sex_ratio, time_step), ratio) in ratios_dict
    push!(ratios_df, (string(sex_ratio), time_step, ratio))
end

# Save the DataFrame as a CSV file
CSV.write("ratios.csv", ratios_df)

1  1  100  67140677
0.0
1  1  75  67140676
0.0
1  1  50  67140675
0.0
1  1  25  67140674
0.0
1  1  10  67140673
0.04
3  1  100  67140737
0.0
3  1  75  67140736
0.0
3  1  50  67140735
0.0
3  1  25  67140734
0.02
3  1  10  67140733
0.32
3  3  100  67140694
0.0
3  3  75  67140693
0.01
3  3  50  67140692
0.0
3  3  25  67140691
0.03
3  3  10  67140689
0.48
3  5  100  67140860
0.0
3  5  75  67140769
0.0
3  5  50  67140768
0.0
3  5  25  67140767
0.02
3  5  10  67140766
0.55
3  10  100  67140782
0.0
3  10  75  67140781
0.01
3  10  50  67140780
0.02
3  10  25  67140779
0.03
3  10  10  67140778
0.56
5  1  100  67140749
0.0
5  1  75  67140748
0.0
5  1  50  67140747
0.02
5  1  25  67140746
0.08
5  1  10  67140745
0.82
5  3  100  67140795
0.0
5  3  75  67140794
0.03
5  3  50  67140793
0.01
5  3  25  67140792
0.15
5  3  10  67140791
0.85
5  5  100  67140703
0.0
5  5  75  67140702
0.02
5  5  50  67140701
0.01
5  5  25  67140700
0.14
5  5  10  67140699
0.93
5  10  100  67140789
0.01
5  10  75  6714078

"ratios.csv"

In [7]:
# Initialize the dictionary to store ratios
ratios_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

# Loop through each combination of sex ratios and release time steps
for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        if JOB_ID != nothing
            df_fem = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "track_fem_m_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_male = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "track_male_m_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            # Calculate the ratio
            df_m = DataFrame([df_fem[:,i] .+ df_male[:,i] for i in 1:ncol(df_fem)], :auto)
            ratio = count_columns_with_zero(df_m) / size(df_m, 2)
            # Store the ratio in the dictionary
            ratios_dict[(r, t)] = ratio
            println(ratio)
        end
    end
end

# Print the dictionary
println(ratios_dict)
# Convert the dictionary to a DataFrame
ratios_df = DataFrame(SexRatio=String[], ReleaseTimeStep=Int[], Ratio=Float64[])
for ((sex_ratio, time_step), ratio) in ratios_dict
    push!(ratios_df, (string(sex_ratio), time_step, ratio))
end

# Save the DataFrame as a CSV file
CSV.write("ratios_single_hhold.csv", ratios_df)

1  1  100  67140677
0.03
1  1  75  67140676
0.07
1  1  50  67140675
0.1
1  1  25  67140674
0.07
1  1  10  67140673
0.25
3  1  100  67140737
0.03
3  1  75  67140736
0.06
3  1  50  67140735
0.12
3  1  25  67140734
0.11
3  1  10  67140733
0.52
3  3  100  67140694
0.07
3  3  75  67140693
0.11
3  3  50  67140692
0.04
3  3  25  67140691
0.16
3  3  10  67140689
0.64
3  5  100  67140860
0.05
3  5  75  67140769
0.07
3  5  50  67140768
0.06
3  5  25  67140767
0.17
3  5  10  67140766
0.72
3  10  100  67140782
0.07
3  10  75  67140781
0.09
3  10  50  67140780
0.09
3  10  25  67140779
0.19
3  10  10  67140778
0.76
5  1  100  67140749
0.08
5  1  75  67140748
0.11
5  1  50  67140747
0.14
5  1  25  67140746
0.25
5  1  10  67140745
0.9
5  3  100  67140795
0.09
5  3  75  67140794
0.12
5  3  50  67140793
0.11
5  3  25  67140792
0.25
5  3  10  67140791
0.94
5  5  100  67140703
0.08
5  5  75  67140702
0.08
5  5  50  67140701
0.16
5  5  25  67140700
0.3
5  5  10  67140699
0.97
5  10  100  67140789
0.07
5  1

"ratios_single_hhold.csv"

In [8]:
wolb_pers_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        #df_track_fem_m = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "track_fem_m_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
        #df_track_male_m = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)","track_male_m_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
        df_track_fem_w = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)","track_fem_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
        df_track_male_w = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)","track_male_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)

        wolb_persist = df_track_fem_w .+ df_track_male_w # sum males and females
        n_rows, n_cols = size(wolb_persist)
        indices_zero_after_non_zero = fill(t_end+1, n_cols)  # Initialize with t_end to indicate no such zero found
        indices_first_non_zero = fill(t_end+1, n_cols)      # Initialize with t_end to indicate no non-zero found

        for col in 1:n_cols  # finds index where wolbachia first introduced and index where it goes extinct (if does)
            found_non_zero = false
            for row in 1:n_rows
                if wolb_persist[row, col] != 0
                    if !found_non_zero
                        indices_first_non_zero[col] = row
                        found_non_zero = true
                    end
                elseif found_non_zero && wolb_persist[row, col] == 0
                    indices_zero_after_non_zero[col] = row
                    break
                end
            end
        end

        wolb_pers_dict[(r, t)] = mean(indices_zero_after_non_zero - indices_first_non_zero)
    end
end

# Convert the dictionary to a DataFrame
sex_ratios_col = [string(k[1]) for k in keys(wolb_pers_dict)]
release_time_steps_col = [k[2] for k in keys(wolb_pers_dict)]
pers_time_col = [v for v in values(wolb_pers_dict)]

wolb_pers_df = DataFrame(SexRatio=sex_ratios_col, ReleaseTimeStep=release_time_steps_col, PersTime=pers_time_col)

# Save the DataFrame as a CSV file
CSV.write("wolb_pers_single_hhold.csv", wolb_pers_df)
# Load the CSV file into a DataFrame
wolb_pers_df_loaded = CSV.read("wolb_pers_single_hhold.csv", DataFrame)

1  1  100  67140677
1  1  75  67140676
1  1  50  67140675
1  1  25  67140674
1  1  10  67140673
3  1  100  67140737
3  1  75  67140736
3  1  50  67140735
3  1  25  67140734
3  1  10  67140733
3  3  100  67140694
3  3  75  67140693
3  3  50  67140692
3  3  25  67140691
3  3  10  67140689
3  5  100  67140860
3  5  75  67140769
3  5  50  67140768
3  5  25  67140767
3  5  10  67140766
3  10  100  67140782
3  10  75  67140781
3  10  50  67140780
3  10  25  67140779
3  10  10  67140778
5  1  100  67140749
5  1  75  67140748
5  1  50  67140747
5  1  25  67140746
5  1  10  67140745
5  3  100  67140795
5  3  75  67140794
5  3  50  67140793
5  3  25  67140792
5  3  10  67140791
5  5  100  67140703
5  5  75  67140702
5  5  50  67140701
5  5  25  67140700
5  5  10  67140699
5  10  100  67140789
5  10  75  67140787
5  10  50  67140786
5  10  25  67140785
5  10  10  67140784
10  3  100  67140803
10  3  75  67140799
10  3  50  67140798
10  3  25  67140797
10  3  10  67140796
10  5  100  67140809
10  

Row,SexRatio,ReleaseTimeStep,PersTime
,String15,Int64,Float64
1,"(3, 10)",75,15.26
2,"(5, 10)",25,25.82
3,"(10, 1)",10,572.13
4,"(10, 10)",10,662.42
5,"(5, 1)",75,17.74
6,"(3, 3)",50,12.91
7,"(5, 10)",75,17.27
8,"(3, 5)",100,14.82
9,"(10, 3)",100,22.18


In [9]:
# Initialize the dictionary to store ratios
wolb_av_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

# Loop through each combination of sex ratios and release time steps
for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        if JOB_ID != nothing
            df_fem = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "track_fem_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_male = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "track_male_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            # Calculate the ratio            
            wolb_av = sum(Array(df_fem[end, :]) .+ Array(df_male[end, :]))/num_simulations
            # Store the ratio in the dictionary
            wolb_av_dict[(r, t)] = wolb_av
            #println(wolb_av)
        end
    end
end

# Print the dictionary
#println(wolb_av_dict)
# Convert the dictionary to a DataFrame
wolb_av_df = DataFrame(SexRatio=String[], ReleaseTimeStep=Int[], WolbAv=Float64[])
for ((sex_ratio, time_step), wolb_av) in wolb_av_dict
    push!(wolb_av_df, (string(sex_ratio), time_step, wolb_av))
end

# Save the DataFrame as a CSV file
CSV.write("wolb_av_single_hhold.csv", wolb_av_df)
println(wolb_av_df)

1  1  100  67140677
1  1  75  67140676
1  1  50  67140675
1  1  25  67140674
1  1  10  67140673
3  1  100  67140737
3  1  75  67140736
3  1  50  67140735
3  1  25  67140734
3  1  10  67140733
3  3  100  67140694
3  3  75  67140693
3  3  50  67140692
3  3  25  67140691
3  3  10  67140689
3  5  100  67140860
3  5  75  67140769
3  5  50  67140768
3  5  25  67140767
3  5  10  67140766
3  10  100  67140782
3  10  75  67140781
3  10  50  67140780
3  10  25  67140779
3  10  10  67140778
5  1  100  67140749
5  1  75  67140748
5  1  50  67140747
5  1  25  67140746
5  1  10  67140745
5  3  100  67140795
5  3  75  67140794
5  3  50  67140793
5  3  25  67140792
5  3  10  67140791
5  5  100  67140703
5  5  75  67140702
5  5  50  67140701
5  5  25  67140700
5  5  10  67140699
5  10  100  67140789
5  10  75  67140787
5  10  50  67140786
5  10  25  67140785
5  10  10  67140784
10  3  100  67140803
10  3  75  67140799
10  3  50  67140798
10  3  25  67140797
10  3  10  67140796
10  5  100  67140809
10  

In [ ]:
av_wolbs_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}() 
sum_w = 0

for r in sex_ratios
    for t in rel_time_steps
        sum_w = 0
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        for i in 1:num_simulations
            fem_w_hholds = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "fem_w_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            male_w_hholds = CSV.read(joinpath("testing_gen_hhold_release", "$(JOB_ID)", "$(JOB_ID)", "male_w_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            sum_w += (sum(male_w_hholds[end,:]) + sum(fem_w_hholds[end,:]))/(H-1)
        end
        wolb_av_dict[(r, t)] = sum_w / num_simulations
    end
end

In [39]:
# Convert the dictionary to a DataFrame
wolb_av_df = DataFrame(SexRatio=String[], ReleaseTimeStep=Int[], WolbAv=Float64[])
for ((sex_ratio, time_step), wolb_av) in wolb_av_dict
    push!(wolb_av_df, (string(sex_ratio), time_step, wolb_av))
end

# Save the DataFrame as a CSV file
CSV.write("wolb_av_all_hholds.csv", wolb_av_df)

"wolb_av_all_hholds.csv"